In [ ]:
!pip install -q transformers torch faiss-cpu Pillow tqdm h5py

In [ ]:
import os, time, numpy as np, torch, torch.nn.functional as F, tarfile
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoProcessor
import faiss, h5py, pickle

# Auto-detect and extract dataset
INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}

def find_and_extract_dataset():
    """Find dataset, extract tar if needed, return path with images"""
    if not INPUT_DIR.exists():
        return None
    for dataset_dir in sorted(INPUT_DIR.iterdir()):
        if not dataset_dir.is_dir():
            continue
        # Check for tar files first
        tar_files = list(dataset_dir.glob("*.tar")) + list(dataset_dir.glob("*.tar.gz"))
        if tar_files:
            extract_dir = WORK_DIR / "dataset"
            extract_dir.mkdir(exist_ok=True)
            print(f"Extracting {tar_files[0].name}...")
            with tarfile.open(tar_files[0], 'r:*') as tar:
                tar.extractall(extract_dir)
            print(f"✓ Extracted to {extract_dir}")
            return str(extract_dir)
        # Check if directory has images directly
        for ext in SUPPORTED_EXTS:
            if list(dataset_dir.rglob(f"*{ext}")):
                return str(dataset_dir)
    return None

DATASET_PATH = find_and_extract_dataset()
if not DATASET_PATH:
    raise FileNotFoundError(f"No dataset found in {INPUT_DIR}")
print(f"✓ Dataset: {DATASET_PATH}")

MODEL_ID = "google/siglip2-base-patch16-naflex"
EMBEDDING_DIM = 768
OUTPUT_DIR = "/kaggle/working"

# Adaptive GPU/CPU configuration
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    
    if "P100" in gpu_name:
        device, BATCH_SIZE, NUM_WORKERS = "cpu", 32, 2
        print(f"GPU: {gpu_name} (not supported) - using CPU")
    elif num_gpus > 1:
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 320, 4
        print(f"GPU: {gpu_name} x{num_gpus} | Batch: {BATCH_SIZE}")
    else:
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 256, 4
        print(f"GPU: {gpu_name} | Batch: {BATCH_SIZE}")
else:
    device, BATCH_SIZE, NUM_WORKERS = "cpu", 32, 2
    print("CPU mode")

In [ ]:
class FastDataset(Dataset):
    def __init__(self, paths, proc):
        self.paths, self.proc = paths, proc
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except:
            img = Image.new("RGB", (224,224), 0)
        inputs = self.proc(images=img, return_tensors="pt")
        return {k: v.squeeze(0) for k, v in inputs.items()}, self.paths[i]

# Scan for all images recursively
all_images = []
for ext in SUPPORTED_EXTS:
    all_images.extend(Path(DATASET_PATH).rglob(f"*{ext}"))

# Store relative paths as IDs (e.g., 'animals/cat/001.jpg')
image_ids = [str(p.relative_to(DATASET_PATH)) for p in all_images]
image_paths = [str(p) for p in all_images]

print(f"✓ Found {len(image_ids)} images")
if image_ids:
    print(f"  Sample: {image_ids[0]}")

In [ ]:
print(f"Loading {MODEL_ID}...")
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)

dtype = torch.float16 if device == "cuda" else torch.float32
use_amp = device == "cuda"
if use_amp: print("  Using FP16")

model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True)

# Remove text encoder to save memory
if hasattr(model, "text_model"): del model.text_model
if hasattr(model, "text_projection"): del model.text_projection
torch.cuda.empty_cache()

class VisionWrapper(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.model = m
    def forward(self, *args, **kwargs):
        out = self.model.get_image_features(*args, **kwargs)
        return out.pooler_output if hasattr(out, 'pooler_output') else out

wrapped = VisionWrapper(model)
if device == "cuda" and torch.cuda.device_count() > 1:
    final_model = torch.nn.DataParallel(wrapped).cuda()
    print("  ✓ DataParallel enabled")
else:
    final_model = wrapped.to(device)

final_model.eval()
if device == "cuda": torch.backends.cudnn.benchmark = True
print(f"✓ Model ready ({time.time()-t0:.1f}s)")

In [ ]:
loader = DataLoader(
    FastDataset(image_paths, processor),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device=="cuda"),
    persistent_workers=(NUM_WORKERS>0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None
)

all_emb, t0 = [], time.time()
with torch.no_grad():
    for inputs, _ in tqdm(loader, desc="Embedding"):
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}
        
        if use_amp:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                feat = final_model(**inputs)
                feat = F.normalize(feat, p=2, dim=-1)
        else:
            feat = final_model(**inputs)
            feat = F.normalize(feat, p=2, dim=-1)
        all_emb.append(feat.cpu().numpy())

if device == "cuda": torch.cuda.synchronize()
embeddings = np.vstack(all_emb)
inf_time = time.time() - t0
print(f"✓ Inference: {inf_time:.1f}s ({len(image_paths)/inf_time:.1f} img/s)")

In [ ]:
# Save embeddings to HDF5
with h5py.File(OUTPUT_DIR + "/embeddings.hdf5", "w") as f:
    f.create_dataset("embeddings", data=embeddings)
    f.create_dataset("image_ids", data=[s.encode("utf-8") for s in image_ids])
    f.create_dataset("image_paths", data=[s.encode("utf-8") for s in image_paths])
print(f"✓ Saved embeddings.hdf5 ({embeddings.nbytes / 1024**2:.1f} MB)")

# Build FAISS index
index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(embeddings.astype(np.float32))
with open(OUTPUT_DIR + "/faiss_index.pkl", "wb") as f:
    pickle.dump(index, f)
print(f"✓ Saved faiss_index.pkl")